# Reto 7: Búsqueda Semántica de Noticias (Parte 1)
**Responsabilidad:** Ingesta, EDA, Limpieza y Preprocesamiento.

De acuerdo a las reglas del curso y a los requerimientos del reto:
1. Usamos el dataset `20 Newsgroups` de scikit-learn.
2. Todo está documentado justificando las decisiones.
3. Se fijará la semilla aleatoria `random_state=42` para reproducibilidad.
4. Exportaremos los datasets finales a CSV para que la fase de Modelado los consuma fácilmente.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split

# Fijar la semilla para que todo sea reproducible
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Configuración visual
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## Fase 1: Ingesta de Datos
Para este reto, el objetivo es tener un conjunto de noticias sobre los cuales haremos búsqueda semántica.
En lugar de depender de un CSV externo que requiera descargas manuales, utilizaremos la librería `scikit-learn` para descargar de forma automática el dataset **20 Newsgroups**, que consiste en miles de documentos clasificados en diversas categorías.

In [ ]:
# Descargamos todos los subconjuntos disponibles (train y test juntos) para tener todo el corpus
# Nota: removemos los metadatos (headers, footers, quotes) desde la ingesta para enfocarnos en el cuerpo de la noticia.
print("Descargando dataset 20 Newsgroups...")
newsgroups = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'), random_state=RANDOM_STATE)

# Convertimos los datos a un DataFrame de Pandas para mayor facilidad de manipulación
df = pd.DataFrame({
    'text': newsgroups.data,
    'target': newsgroups.target
})

# Agregamos la categoría en texto legible
df['category'] = df['target'].apply(lambda x: newsgroups.target_names[x])

print(f"Total de documentos ingeridos: {len(df)}")
display(df.head())

## Fase 2: Análisis Exploratorio de Datos (EDA)
Vamos a analizar cómo son nuestros datos. 
Las preguntas que queremos responder son:
1. ¿Están balanceadas las categorías de nuestras noticias?
2. ¿Cómo se distribuye la longitud de los textos? (Esto es clave para los modelos Transformer que suelen tener un límite de 512 tokens).

In [ ]:
# 1. Distribución de categorías
plt.figure(figsize=(12, 8))
sns.countplot(y='category', data=df, order=df['category'].value_counts().index, palette='viridis')
plt.title('Distribución de Documentos por Categoría', fontsize=16)
plt.xlabel('Cantidad de Documentos')
plt.ylabel('Categoría')
plt.show()

# Conclusión 1: El dataset está bastante balanceado, casi todas las categorías tienen alrededor de 1000 documentos.

In [ ]:
# 2. Distribución de la longitud de los textos
# Contamos el número de palabras por documento aproximado separando por espacios
df['word_count'] = df['text'].apply(lambda x: len(str(x).split()))

plt.figure(figsize=(12, 6))
sns.histplot(df['word_count'], bins=100, kde=True, color='blue')
plt.title('Distribución de la cantidad de palabras por documento', fontsize=16)
plt.xlabel('Cantidad de Palabras')
plt.ylabel('Frecuencia')
plt.xlim(0, 2000) # Recortamos a 2000 para mejor visualización
plt.show()

# Estadísticas descriptivas
print("Estadísticas descriptivas de la cantidad de palabras:")
display(df['word_count'].describe())

# Conclusión 2: La mediana es de ~91 palabras, pero tenemos outliers inmensos.
# Para Sentence-Transformers (limitados a 512 tokens usualmente), truncar a las primeras palabras será suficiente.

## Fase 3: Limpieza
Vamos a aplicar las siguientes reglas:
1. **Valores nulos / vacíos:** Eliminar filas donde el texto esté vacío o tenga muy pocas palabras (por ejemplo, menos de 5 palabras no sirven para la búsqueda semántica).
2. **Expresiones regulares (RegEx):** Limpiar caracteres extraños, símbolos raros y pasar todo a minúsculas para unificar.

In [ ]:
# 1. Filtrar textos demasiado cortos (menos de 5 palabras)
initial_len = len(df)
df = df[df['word_count'] > 5].copy()
print(f"Documentos eliminados por ser muy cortos: {initial_len - len(df)}")

# 2. Función de limpieza de texto
def clean_text(text):
    text = str(text).lower() # Convertir a minúsculas
    text = re.sub(r'\s+', ' ', text) # Reemplazar múltiples espacios/saltos de línea por uno solo
    text = re.sub(r'[^a-z0-9áéíóúñü\.,!?;:()\- ]', '', text) # Eliminar caracteres extraños pero mantener puntuación básica
    return text.strip()

# Aplicar limpieza
print("Aplicando limpieza de texto...")
df['clean_text'] = df['text'].apply(clean_text)

# Validamos cómo quedó un documento al azar
print("\nEjemplo de texto limpio:")
print(df['clean_text'].iloc[0][:500])

## Fase 4: Preprocesamiento
De acuerdo a las reglas de la unidad, necesitamos:
1. Dividir en **Train (80%), Validation (10%) y Test (10%)**.
2. Guardar estos conjuntos para que la persona de "Modelado" los ingiera directamente.

In [ ]:
# Dividimos primero en Train (80%) y Temporal (20%)
X = df['clean_text']
y = df['category']

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y)

# Dividimos el Temporal en Validation (50% de 20% = 10%) y Test (50% de 20% = 10%)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp)

print(f"Tamaño del conjunto de Entrenamiento (Train): {len(X_train)} ({len(X_train)/len(df)*100:.1f}%)")
print(f"Tamaño del conjunto de Validación (Val): {len(X_val)} ({len(X_val)/len(df)*100:.1f}%)")
print(f"Tamaño del conjunto de Prueba (Test): {len(X_test)} ({len(X_test)/len(df)*100:.1f}%)")

In [ ]:
# Empaquetamos en DataFrames para exportar
train_df = pd.DataFrame({'text': X_train, 'category': y_train})
val_df = pd.DataFrame({'text': X_val, 'category': y_val})
test_df = pd.DataFrame({'text': X_test, 'category': y_test})

# Exportamos a CSV local
import os
os.makedirs("datos_preprocesados", exist_ok=True)

train_df.to_csv("datos_preprocesados/train.csv", index=False)
val_df.to_csv("datos_preprocesados/val.csv", index=False)
test_df.to_csv("datos_preprocesados/test.csv", index=False)

print("¡Archivos exportados exitosamente en la carpeta 'datos_preprocesados'!")
print("El equipo de Modelado ya puede comenzar su trabajo usando pd.read_csv('datos_preprocesados/train.csv')")